### testing if there exits

In [2]:
import pandas as pd
import ollama
from psycopg2.extras import execute_values
from database import get_connection

In [3]:
CSV_FILE = "../../../hr_interview_questions_cleaned.csv"
EMBEDDING_MODEL = "nomic-embed-text"
# Number of rows processed by PostgreSQL at once
DB_BATCH_SIZE = 1000
# Number of texts sent to Ollama in one embedding request
EMBED_BATCH_SIZE = 500
# Keep model loaded in memory
KEEP_ALIVE = "10m"

In [25]:
df = pd.read_csv(CSV_FILE)
total_rows = len(df)

In [5]:
df.head(2)

,question,category,role,experience,difficulty,source_type,ideal_answer,keywords
0,Tell me about a time you had to learn somethin...,Adaptability,DevOps Engineer,fresher,Easy,Open-Ended,I'm always eager to learn and embrace change a...,"['flexible', 'change']"
1,Describe a time you handled a difficult situat...,Conflict Resolution,Product Manager,2 years,Hard,Behavioral,"When faced with conflict, I approach it calmly...","['problem', 'disagree']"


In [27]:
conn = get_connection()
cur = conn.cursor()

In [28]:
cur.execute("""
        SELECT COUNT(*)
        FROM interview_questions;
    """)

In [29]:
already_imported = cur.fetchone()[0]

print(
    f"Already in database: "
    f"{already_imported:,}"
)

if already_imported >= total_rows:

    print("\nAll rows are already imported.")

    cur.close()
    conn.close()
    return

Already in database: 146,200


SyntaxError: 'return' outside function (2972744690.py, line 14)

In [30]:
df = df.iloc[already_imported:]

In [31]:
df.head(2)

,question,category,role,experience,difficulty,source_type,ideal_answer,keywords
146200,Tell me about a time you mentored someone.,Leadership,UX Designer,5 years,Easy,Situational,I lead by example and encourage open communica...,"['guide', 'lead']"
146201,Describe a time you led a project or team.,Leadership,HR Specialist,3 years,Easy,Behavioral,I lead by example and encourage open communica...,"['supervise', 'mentor']"


In [32]:
remaining = len(df)
remaining

594062

In [21]:
def clean_value(value):

    if pd.isna(value):
        return ""

    return str(value)

In [22]:
def create_embedding_text(row):

    question = clean_value(row.question)
    category = clean_value(row.category)
    role = clean_value(row.role)
    experience = clean_value(row.experience)
    difficulty = clean_value(row.difficulty)
    source_type = clean_value(row.source_type)
    keywords = clean_value(row.keywords)

    # Keep ideal_answer out of the embedding.
    # It is stored in PostgreSQL and retrieved after similarity search.

    return f"""
Question: {question}
Category: {category}
Role: {role}
Experience: {experience}
Difficulty: {difficulty}
Type: {source_type}
Keywords: {keywords}
"""

In [23]:
def create_embeddings(texts):

    response = ollama.embed(
        model=EMBEDDING_MODEL,
        input=texts,
        keep_alive=KEEP_ALIVE
    )

    return response["embeddings"]

In [33]:
def process_batch(rows):

    # Create embedding text for every row
    texts = [
        create_embedding_text(row)
        for row in rows
    ]

    # Send multiple texts to Ollama in one request
    embeddings = create_embeddings(texts)

    records = []

    for row, embedding in zip(rows, embeddings):

        records.append(
            (
                clean_value(row.question),
                clean_value(row.category),
                clean_value(row.role),
                clean_value(row.experience),
                clean_value(row.difficulty),
                clean_value(row.source_type),
                clean_value(row.ideal_answer),
                clean_value(row.keywords),
                embedding
            )
        )

    return records

In [34]:
for start in range(
    0,
    remaining,
    DB_BATCH_SIZE
):

    batch = df.iloc[
        start:start + DB_BATCH_SIZE
    ]

    first_row = (
        already_imported
        + start
        + 1
    )

    last_row = (
        already_imported
        + start
        + len(batch)
    )

    print(
        f"Processing rows "
        f"{first_row:,} - "
        f"{last_row:,}"
    )

    # Convert dataframe rows to tuples
    rows = list(
        batch.itertuples(
            index=False
        )
    )

    all_records = []

    # Generate embeddings in smaller groups
    for embed_start in range(
        0,
        len(rows),
        EMBED_BATCH_SIZE
    ):

        embed_rows = rows[
            embed_start:
            embed_start + EMBED_BATCH_SIZE
        ]

        print(
            f"  Embedding "
            f"{first_row + embed_start:,} - "
            f"{min(first_row + embed_start + len(embed_rows) - 1, last_row):,}"
        )

        records = process_batch(
            embed_rows
        )

        all_records.extend(
            records
        )

    # Insert the complete database batch
    execute_values(
        cur,

        """
        INSERT INTO interview_questions (
            question,
            category,
            role,
            experience,
            difficulty,
            source_type,
            ideal_answer,
            keywords,
            embedding
        )
        VALUES %s
        """,

        all_records,

        page_size=500
    )

    conn.commit()

    total_imported = (
        already_imported
        + start
        + len(batch)
    )

    percentage = (
        total_imported
        / total_rows
    ) * 100

    print(
        f"Database progress: "
        f"{total_imported:,}/"
        f"{total_rows:,} "
        f"({percentage:.2f}%)"
    )

    print("Batch saved.\n")

cur.close()
conn.close()

print("=" * 60)
print("IMPORT COMPLETED")
print("=" * 60)

Processing rows 146,201 - 147,200
  Embedding 146,201 - 146,700
  Embedding 146,701 - 147,200
Database progress: 147,200/740,262 (19.88%)
Batch saved.

Processing rows 147,201 - 148,200
  Embedding 147,201 - 147,700
  Embedding 147,701 - 148,200
Database progress: 148,200/740,262 (20.02%)
Batch saved.

Processing rows 148,201 - 149,200
  Embedding 148,201 - 148,700
  Embedding 148,701 - 149,200
Database progress: 149,200/740,262 (20.16%)
Batch saved.

Processing rows 149,201 - 150,200
  Embedding 149,201 - 149,700
  Embedding 149,701 - 150,200
Database progress: 150,200/740,262 (20.29%)
Batch saved.

Processing rows 150,201 - 151,200
  Embedding 150,201 - 150,700
  Embedding 150,701 - 151,200
Database progress: 151,200/740,262 (20.43%)
Batch saved.

Processing rows 151,201 - 152,200
  Embedding 151,201 - 151,700
  Embedding 151,701 - 152,200
Database progress: 152,200/740,262 (20.56%)
Batch saved.

Processing rows 152,201 - 153,200
  Embedding 152,201 - 152,700
  Embedding 152,701 - 15

KeyboardInterrupt: 